# Phase 0 — Sélection du Modèle VLM / VLM Model Selection Study
**Projet / Project:** Système d'Inspection des Ouvrages d'Art par IA (CID)  
**Auteur / Author:** Youssef Elhardouzi  
**Date:** Avril 2026  
**Version:** v5 — Évaluation étendue à TOUS les modèles mentionnés (8 modèles)

---

## Méthodologie de sélection / Selection Methodology

Les candidats ont été sélectionnés en deux phases :

**Phase initiale** — Open VLM Leaderboard (OpenCompass, consultation Mars 2026), top 15, filtrés par :
1. Paramètres ≤ 4B (contrainte VRAM T4)
2. Disponibilité publique sur HuggingFace Hub
3. Interface standard AutoModel
4. Support de fine-tuning confirmé dans LLaMA-Factory
5. Licence ouverte pour usage commercial (Apache 2.0 ou MIT)

**Addendum Phase 0** — Révision systématique des modèles LLaMA-Factory face aux sorties récentes HuggingFace (Mars–Avril 2026) avec les mêmes critères.

**Modèles exclus avec justification :**
- `SmolVLM2-2.2B` — pas de template LLaMA-Factory (évalué ici à titre informatif)
- `Gemma3-4B` — 4B à la limite VRAM T4 ; licence Gemma Terms of Use (non‑vision, ignoré)
- `Qwen2.5-Omni-3B` — licence qwen-research (recherche uniquement)
- `Qwen3.5-VL` — n'existe pas (Qwen3.5 est texte uniquement)
- `LFM2.5-VL` — licence LFM 1.0 (restreint aux entreprises < 10M$ CA)

**Candidats retenus (4) + modèles exclus (5) → évaluation complète :**

| Modèle | Params | Licence | HF path | Famille |
|--------|--------|---------|---------|---------|
| InternVL3-2B | 2B | MIT | OpenGVLab/InternVL3-2B-hf | internvl |
| Qwen2.5-VL-3B | 3B | Apache 2.0 | Qwen/Qwen2.5-VL-3B-Instruct | qwen_vl |
| Qwen3-VL-2B | 2B | Apache 2.0 | Qwen/Qwen3-VL-2B-Instruct | qwen_vl |
| InternVL3.5-2B | 2.35B | Apache 2.0 | OpenGVLab/InternVL3_5-2B-HF | internvl |
| SmolVLM2-2.2B | 2.2B | Apache 2.0 | HuggingFaceTB/SmolVLM2-2.2B-Instruct | smolvlm |
| Gemma3-4B* | 4B | Gemma TOS | google/gemma-3-4b-it | gemma (text only) |
| Qwen2.5-Omni-3B | 3B | qwen-research | Qwen/Qwen2.5-Omni-3B | qwen_omni |
| LFM2.5-VL | 4B? | LFM 1.0 | liquid/LFM-2.5-VL-4B | lfm_vl |

*Gemma3-4B n’est pas un modèle vision‑langage ; l’évaluation échouera volontairement.*

---
## 0 — Installation

Installe toutes les dépendances. À exécuter une seule fois.  
**Redémarrez le kernel manuellement après cette cellule.**

In [ ]:
import subprocess, sys

packages = [
    'transformers>=4.57.0',
    'accelerate>=0.29.0',
    'bitsandbytes>=0.43.0',
    'qwen-vl-utils',
    'pillow',
    'torchvision',
    'pandas',
    'rouge-score',
    'nltk',
    'python-dotenv',
    'num2words'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✓ Installation terminée.')
print('IMPORTANT: Redémarrez le kernel maintenant avant de continuer.')

✓ Installation terminée.
IMPORTANT: Redémarrez le kernel maintenant avant de continuer.


---
## 1 — Configuration

In [ ]:
import os, sys, json, gc, time, base64, textwrap, warnings, random
import pandas as pd
import torch
from io import BytesIO
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ── Mount Google Drive (Colab only) ───────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive monté')
except ImportError:
    pass

# ─────────────────────────────────────────────────────────────────────────
# EDIT THESE if you don't use .env / Secrets
HF_TOKEN_OVERRIDE = ''
CID_BASE_OVERRIDE = ''   # e.g. '/content/drive/MyDrive/CID'
# ─────────────────────────────────────────────────────────────────────────

def _get_secret(key):
    overrides = {'HF_TOKEN': HF_TOKEN_OVERRIDE, 'CID_BASE': CID_BASE_OVERRIDE}
    if overrides.get(key): return overrides[key]
    if os.environ.get(key): return os.environ[key]
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception: pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception: pass
    return None

# ── HuggingFace token ──────────────────────────────────────────────────────
HF_TOKEN = _get_secret('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HF_TOKEN : authentifié')
else:
    print('⚠ HF_TOKEN non trouvé — modèles publics uniquement')

# ── Resolve paths ─────────────────────────────────────────────────────────
CID_BASE = _get_secret('CID_BASE')
if not CID_BASE:
    notebook_dir = os.path.abspath('.')
    CID_BASE = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) in ('phase0', 'notebooks') else notebook_dir
    print(f'⚠ CID_BASE non défini — auto-détecté : {CID_BASE}')
else:
    print(f'✓ CID_BASE : {CID_BASE}')

JSON_DATA    = os.path.join(CID_BASE, 'data', 'finetuning_dataset.json')
RESULTS_DIR  = os.path.join(CID_BASE, 'phase0')
results_path = os.path.join(RESULTS_DIR, 'model_selection_results_v5.json')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── GPU check ──────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), 'Aucun GPU détecté.'
print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import transformers; print(f'✓ Transformers : {transformers.__version__}')

# ── RAM tracking for CPU ────────────────────────────────────────────────
import psutil
import os

def get_ram_usage_gb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1e9

# ── Inspection prompt ──────────────────────────────────────────────────────
INSTRUCTION = (
    "Tu es un expert en inspection d'ouvrages d'art. "
    "Observe attentivement cette image et génère uniquement le titre technique du désordre visible, "
    "en français, en une phrase courte. Ne génère rien d'autre."
)

# ─────────────────────────────────────────────────────────────────────────
# MODEL REGISTRY — TOUS les modèles mentionnés dans l'introduction
# ─────────────────────────────────────────────────────────────────────────
MODEL_REGISTRY = [
    # Candidats retenus (4)
    {
        'id':      'InternVL3-2B',
        'hf':      'OpenGVLab/InternVL3-2B-hf',
        'family':  'internvl',
        'lf_tpl':  'intern_vl',
        'license': 'MIT',
        'rank':    10,
        'avg':     64.5,
    },
    {
        'id':      'Qwen2.5-VL-3B',
        'hf':      'Qwen/Qwen2.5-VL-3B-Instruct',
        'family':  'qwen_vl',
        'lf_tpl':  'qwen2_5_vl',
        'license': 'Apache 2.0',
        'rank':    11,
        'avg':     64.5,
    },
    {
        'id':      'Qwen3-VL-2B',
        'hf':      'Qwen/Qwen3-VL-2B-Instruct',
        'family':  'qwen_vl',
        'lf_tpl':  'qwen3_vl',
        'license': 'Apache 2.0',
        'rank':    None,
        'avg':     None,
    },
    {
        'id':      'InternVL3.5-2B',
        'hf':      'OpenGVLab/InternVL3_5-2B-HF',
        'family':  'internvl',
        'lf_tpl':  'intern_vl',
        'license': 'Apache 2.0',
        'rank':    None,
        'avg':     None,
    },
    # Modèles exclus (5) — évalués malgré les restrictions
    {
        'id':      'SmolVLM2-2.2B',
        'hf':      'HuggingFaceTB/SmolVLM2-2.2B-Instruct',
        'family':  'smolvlm',
        'lf_tpl':  None,   # pas de template LLaMA-Factory
        'license': 'Apache 2.0',
        'rank':    None,
        'avg':     None,
    },
    {
        'id':      'Gemma3-4B',
        'hf':      'google/gemma-3-4b-it',
        'family':  'gemma',   # texte uniquement → échouera
        'lf_tpl':  None,
        'license': 'Gemma Terms of Use',
        'rank':    None,
        'avg':     None,
    },
    {
        'id':      'Qwen2.5-Omni-3B',
        'hf':      'Qwen/Qwen2.5-Omni-3B',
        'family':  'qwen_omni',
        'lf_tpl':  None,
        'license': 'qwen-research',
        'rank':    None,
        'avg':     None,
    },
    {
        'id':      'LFM2.5-VL',
        'hf':      'LiquidAI/LFM2.5-VL-1.6B',
        'family':  'lfm_vl',
        'lf_tpl':  None,
        'license': 'LFM 1.0',
        'rank':    None,
        'avg':     None,
    },
]

print(f'\n✓ Configuration chargée')
print(f'  CID_BASE    : {CID_BASE}')
print(f'  JSON data   : {JSON_DATA}')
print(f'  Results dir : {RESULTS_DIR}')
print(f'  Modèles     : {[m["id"] for m in MODEL_REGISTRY]}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive monté
✓ HF_TOKEN : authentifié
✓ CID_BASE : /content/drive/MyDrive/CID
✓ GPU  : Tesla T4
✓ VRAM : 15.6 GB
✓ Transformers : 5.0.0

✓ Configuration chargée
  CID_BASE    : /content/drive/MyDrive/CID
  JSON data   : /content/drive/MyDrive/CID/data/finetuning_dataset.json
  Results dir : /content/drive/MyDrive/CID/phase0
  Modèles     : ['InternVL3-2B', 'Qwen2.5-VL-3B', 'Qwen3-VL-2B', 'InternVL3.5-2B', 'SmolVLM2-2.2B', 'Gemma3-4B', 'Qwen2.5-Omni-3B', 'LFM2.5-VL']


---
## 2 — Chargement du Dataset et sélection des 10 images

Les 10 images sont sélectionnées de manière stratifiée par famille de désordre,
pour couvrir l'ensemble du spectre ITSEOA plutôt que de tirer aléatoirement.

In [ ]:
if not os.path.exists(JSON_DATA):
    raise FileNotFoundError(f'Dataset introuvable : {JSON_DATA}')

with open(JSON_DATA, encoding='utf-8') as f:
    records = json.load(f)
print(f'✓ Dataset : {len(records)} entrées')

import random
from collections import defaultdict
random.seed(42)

prefix_to_indices = defaultdict(list)
for idx, rec in enumerate(records):
    fiche = rec.get('meta', {}).get('fiche', '')
    prefix = fiche.split('.')[0] if fiche else 'unknown'
    prefix_to_indices[prefix].append(idx)

n_total = 10
test_indices = []
for p in list(prefix_to_indices.keys()):
    if len(test_indices) < n_total and prefix_to_indices[p]:
        chosen = random.choice(prefix_to_indices[p])
        test_indices.append(chosen)
        prefix_to_indices[p].remove(chosen)

remaining = [idx for indices in prefix_to_indices.values() for idx in indices]
if len(test_indices) < n_total:
    extra = random.sample(remaining, min(n_total - len(test_indices), len(remaining)))
    test_indices.extend(extra)
test_indices.sort()

# reference = meta.defect (titre court — nouvelle tâche Phase 0 v5)
test_samples = []
for idx in test_indices:
    rec    = records[idx]
    meta   = rec.get('meta', {})
    fiche  = meta.get('fiche', '?')
    defect = meta.get('defect', '?')
    prefix = fiche.split('.')[0] if fiche != '?' else '?'
    img    = Image.open(BytesIO(base64.b64decode(rec.get('image_b64', '')))).convert('RGB')
    test_samples.append({
        'idx':       idx,
        'fiche':     fiche,
        'family':    prefix,
        'desc':      defect,
        'image':     img,
        'reference': defect,
    })
    print(f'  [{idx:>3}] Fiche {fiche} — {defect[:65]}')

print(f'\n✓ {len(test_samples)} images chargées | référence = meta.defect (titre court)')


✓ Dataset : 214 entrées
Images sélectionnées :
  [ 20] Fiche 2.12 | défaut: Déformations, ruptures, absence des lisses et/ou des montants
  [ 31] Fiche 3.1 | défaut: . Déformation ou absence de grille ou de caillebotis
  [ 36] Fiche 4.1 | défaut: Les éclats résultant d’un choc
  [100] Fiche 5.4 | défaut: Affaissement du corps du trottoir
  [117] Fiche 7.2 | défaut: Écoulement en intrados de tablier
  [143] Fiche 8.8 | défaut: Altération du solin
  [179] Fiche 8.24 | défaut: Ouverture anormale des joints de trottoir (blocage)
  [190] Fiche 9.2 | défaut: Altération ou décollement des relevés en rive
  [196] Fiche 10.1 | défaut: Absence de perré
  [199] Fiche 10.2 | défaut: Perré mal aménagé (absence de plate forme en tête du perré)
  Chargée image 20 : Déformations, ruptures, absence des lisses et/ou des montants
  Chargée image 31 : . Déformation ou absence de grille ou de caillebotis
  Chargée image 36 : Les éclats résultant d’un choc
  Chargée image 100 : Affaissement du corps du trot

---
## 3 — ModelWrapper : interface unifiée (supporte internvl, qwen_vl, smolvlm, qwen_omni, lfm_vl)

In [ ]:
from transformers import (
    BitsAndBytesConfig, AutoProcessor, AutoModelForImageTextToText,
    AutoModelForCausalLM, AutoTokenizer
)

BNB_4BIT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def vram_used_gb():
    return torch.cuda.memory_allocated() / 1e9


class ModelWrapper:
    def __init__(self, entry: dict, device='cuda'):
        self.id = entry['id']
        self.hf = entry['hf']
        self.family = entry['family']
        self.device = device
        self.model = None
        self.proc = None

    def load(self):
        self.proc = AutoProcessor.from_pretrained(self.hf, trust_remote_code=True)

        if self.device == 'cpu':
            # InternVL: fp16 to save RAM (~4.7GB vs ~9.4GB); Qwen needs fp32
            cpu_dtype = torch.float16 if self.family == 'internvl' else torch.float32
            self.model = AutoModelForImageTextToText.from_pretrained(
                self.hf,
                torch_dtype=cpu_dtype,
                device_map='cpu',
                trust_remote_code=True,
            )
        else:
            # GPU: 4-bit quantization
            self.model = AutoModelForImageTextToText.from_pretrained(
                self.hf,
                quantization_config=BNB_4BIT,
                device_map='auto',
                trust_remote_code=True,
            )
        self.model.eval()

    def generate(self, image: Image.Image, prompt: str, max_new_tokens: int = 30) -> str:
        inputs = self._format_inputs(image, prompt)
        # LFM2.5-VL requires float inputs — cast pixel_values if present
        if 'pixel_values' in inputs and inputs['pixel_values'] is not None:
            inputs['pixel_values'] = inputs['pixel_values'].to(
                dtype=next(self.model.parameters()).dtype
            )
        with torch.no_grad():
            ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        n_input = inputs['input_ids'].shape[1]
        return self.proc.decode(ids[0][n_input:], skip_special_tokens=True)

    def _format_inputs(self, image: Image.Image, prompt: str) -> dict:
        messages = [{
            'role': 'user',
            'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}],
        }]

        if self.family == 'internvl':
            text   = self.proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = self.proc(text=[text], images=[image], return_tensors='pt')
            return {k: v.to(self.model.device) for k, v in inputs.items()}

        elif self.family == 'qwen_vl':
            from qwen_vl_utils import process_vision_info
            text       = self.proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            imgs, vids = process_vision_info(messages)
            inputs     = self.proc(
                text=[text], images=imgs, videos=vids,
                padding=True, return_tensors='pt',
            )
            return {k: v.to(self.model.device) for k, v in inputs.items()}

        elif self.family == 'smolvlm':
            text   = self.proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = self.proc(text=[text], images=[image], return_tensors='pt')
            return {k: v.to(self.model.device) for k, v in inputs.items()}

        else:
            # fallback for unknown families (qwen_omni, lfm_vl)
            text   = self.proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = self.proc(text=[text], images=[image], return_tensors='pt')
            return {k: v.to(self.model.device) for k, v in inputs.items()}

    def unload(self):
        try:
            del self.model
            del self.proc
        except Exception:
            pass
        self.model = None
        self.proc  = None
        free_gpu()


print('✓ ModelWrapper initialisé (supporte internvl, qwen_vl, smolvlm, qwen_omni, lfm_vl)')

✓ ModelWrapper initialisé (supporte internvl, qwen_vl, smolvlm, qwen_omni, lfm_vl)


---
## 4 — Boucle d'évaluation / Evaluation Loop (TOUS les modèles)

Résultats sauvegardés après chaque modèle. Reprise automatique si interruption.

> ⚠ **Durée estimée : 3–4 heures** sur T4 Colab gratuit (8 modèles × 10 images).

In [ ]:
# ── Boucle d'évaluation sur GPU puis CPU ─────────────────────────────────

results_all = {}

for device in ['cuda', 'cpu']:
    print(f'\n{"█"*65}')
    print(f'  ÉVALUATION SUR {device.upper()}')
    print(f'{"█"*65}')

    # Different result file for each device
    results_path_dev = results_path.replace('.json', f'_{device}.json')

    if os.path.exists(results_path_dev):
        with open(results_path_dev) as f:
            results = json.load(f)
        done_ids = {r['model_id'] for r in results if r['status'] == 'success'}
        print(f'✓ Résultats existants pour {device} : {len(results)} entrées')
        print(f'  Déjà évalués : {done_ids or "aucun"}')
    else:
        results, done_ids = [], set()

    # On CPU: only run the 4 retained candidates to avoid RAM crash
    registry_for_device = MODEL_REGISTRY if device == 'cuda' else [m for m in MODEL_REGISTRY if m['lf_tpl'] is not None]

    for entry in registry_for_device:
        mid = entry['id']
        if mid in done_ids:
            print(f'\n  ↩ {mid} — déjà évalué, ignoré'); continue

        print(f'\n{"="*65}')
        print(f'  Évaluation : {mid} sur {device.upper()}')
        print(f'  HF         : {entry["hf"]}')
        print(f'{"="*65}')

        result = {
            'model_id':         mid,
            'hf':               entry['hf'],
            'family':           entry['family'],
            'lf_template':      entry['lf_tpl'],
            'license':          entry['license'],
            'leaderboard_rank': entry['rank'],
            'leaderboard_avg':  entry['avg'],
            'device':           device,
            'vram_gb':          None,    # for GPU
            'ram_gb':           None,    # for CPU (load)
            'peak_ram_gb':      None,    # for CPU (inference peak)
            'avg_time_s':       None,
            'avg_tok_per_s':    None,
            'outputs':          [],
            'error':            None,
            'status':           'failed',
        }

        # Skip text-only models
        if entry['family'] in ('gemma', 'qwen_omni', 'lfm_vl'):
            print(f'  ⏭ {mid} — modèle texte uniquement, ignoré')
            result['error'] = 'text-only model, skipped'
            result['status'] = 'skipped'
            results = [r for r in results if r['model_id'] != mid] + [result]
            with open(results_path_dev, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            continue

        wrapper = ModelWrapper(entry, device=device)

        try:
            if device == 'cuda':
                free_gpu()
                vram_before = vram_used_gb()
                ram_before = 0
            else:
                vram_before = 0
                ram_before = get_ram_usage_gb()

            print(f'  Chargement...')
            t0 = time.time()
            wrapper.load()
            load_time = time.time() - t0

            if device == 'cuda':
                result['vram_gb'] = round(vram_used_gb() - vram_before, 2)
                print(f'  ✓ Chargé en {load_time:.0f}s · VRAM : {result["vram_gb"]} GB')
            else:
                ram_after_load = get_ram_usage_gb()
                result['ram_gb'] = round(ram_after_load - ram_before, 2)
                print(f'  ✓ Chargé en {load_time:.0f}s · RAM modèles : {result["ram_gb"]} GB')
                peak_ram = ram_after_load

            times, tok_rates = [], []
            for s in test_samples:
                print(f'  ── [{s["idx"]:>3}] {s["family"]} — {s["desc"][:50]}')
                t0 = time.time()
                out = wrapper.generate(s['image'], INSTRUCTION)
                elapsed = round(time.time() - t0, 1)
                tok_rate = round(len(out.split()) / max(elapsed, 0.1), 1)
                times.append(elapsed); tok_rates.append(tok_rate)

                if device == 'cpu':
                    current_ram = get_ram_usage_gb()
                    if current_ram > peak_ram:
                        peak_ram = current_ram

                result['outputs'].append({
                    'idx': s['idx'],
                    'fiche': s['fiche'],
                    'family': s['family'],
                    'desc': s['desc'],
                    'output': out,
                    'reference': s['reference'],
                    'time_s': elapsed,
                    'tok_per_s': tok_rate,
                })
                print(f'     {elapsed}s · {tok_rate} tok/s · sortie: {out[:80]}...')

            if device == 'cpu':
                result['peak_ram_gb'] = round(peak_ram - ram_before, 2)

            result['avg_time_s'] = round(sum(times) / len(times), 1)
            result['avg_tok_per_s'] = round(sum(tok_rates) / len(tok_rates), 1)
            result['status'] = 'success'
            print(f'\n  ✓ Temps moyen : {result["avg_time_s"]}s · {result["avg_tok_per_s"]} tok/s')

        except Exception as e:
            result['error'] = str(e)
            print(f'  ✗ ÉCHEC : {e}')
        finally:
            wrapper.unload()

        results = [r for r in results if r['model_id'] != mid] + [result]
        with open(results_path_dev, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f'  ✓ Sauvegardé : {results_path_dev}')

    n_ok = sum(1 for r in results if r['status'] == 'success')
    print(f'\n{device.upper()} terminé — {n_ok}/{len(MODEL_REGISTRY)} modèles réussis')
    results_all[device] = results

print(f'\n{"█"*65}')
print('ÉVALUATION COMPLÈTE (GPU + CPU)')


█████████████████████████████████████████████████████████████████
  ÉVALUATION SUR CUDA
█████████████████████████████████████████████████████████████████
✓ Résultats existants pour cuda : 8 entrées
  Déjà évalués : {'SmolVLM2-2.2B', 'Qwen2.5-VL-3B', 'InternVL3-2B', 'InternVL3.5-2B', 'Qwen3-VL-2B', 'Gemma3-4B'}

  ↩ InternVL3-2B — déjà évalué, ignoré

  ↩ Qwen2.5-VL-3B — déjà évalué, ignoré

  ↩ Qwen3-VL-2B — déjà évalué, ignoré

  ↩ InternVL3.5-2B — déjà évalué, ignoré

  ↩ SmolVLM2-2.2B — déjà évalué, ignoré

  ↩ Gemma3-4B — déjà évalué, ignoré

  Évaluation : Qwen2.5-Omni-3B sur CUDA
  HF         : Qwen/Qwen2.5-Omni-3B
  Chargement...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


  ✗ ÉCHEC : Unrecognized configuration class <class 'transformers.models.qwen2_5_omni.configuration_qwen2_5_omni.Qwen2_5OmniConfig'> for this kind of AutoModel: AutoModelForImageTextToText.
Model type should be one of AriaConfig, AyaVisionConfig, BlipConfig, Blip2Config, ChameleonConfig, Cohere2VisionConfig, DeepseekVLConfig, DeepseekVLHybridConfig, Emu3Config, Ernie4_5_VL_MoeConfig, EvollaConfig, FastVlmConfig, Florence2Config, FuyuConfig, Gemma3Config, Gemma3nConfig, GitConfig, Glm46VConfig, Glm4vConfig, Glm4vMoeConfig, GotOcr2Config, IdeficsConfig, Idefics2Config, Idefics3Config, InstructBlipConfig, InstructBlipVideoConfig, InternVLConfig, JanusConfig, Kosmos2Config, Kosmos2_5Config, Lfm2VlConfig, LightOnOcrConfig, Llama4Config, LlavaConfig, LlavaNextConfig, LlavaNextVideoConfig, LlavaOnevisionConfig, Mistral3Config, MllamaConfig, Ovis2Config, PaddleOCRVLConfig, PaliGemmaConfig, PerceptionLMConfig, Pix2StructConfig, PixtralVisionConfig, Qwen2_5_VLConfig, Qwen2VLConfig, Qwen3VLConfig

Loading weights:   0%|          | 0/589 [00:00<?, ?it/s]

Lfm2VlForConditionalGeneration LOAD REPORT from: LiquidAI/LFM2.5-VL-1.6B
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✓ Chargé en 21s · VRAM : 1.3 GB
  ── [ 20] 2 — Déformations, ruptures, absence des lisses et/ou d
  ✗ ÉCHEC : data set to a tensor that requires gradients must be floating point or complex dtype
  ✓ Sauvegardé : /content/drive/MyDrive/CID/phase0/model_selection_results_v5_cuda.json

CUDA terminé — 6/8 modèles réussis

█████████████████████████████████████████████████████████████████
  ÉVALUATION SUR CPU
█████████████████████████████████████████████████████████████████
✓ Résultats existants pour cpu : 1 entrées
  Déjà évalués : {'InternVL3-2B'}

  ↩ InternVL3-2B — déjà évalué, ignoré

  Évaluation : Qwen2.5-VL-3B sur CPU
  HF         : Qwen/Qwen2.5-VL-3B-Instruct
  Chargement...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✓ Chargé en 38s · RAM modèles : 7.54 GB
  ── [ 20] 2 — Déformations, ruptures, absence des lisses et/ou d
     184.9s · 0.1 tok/s · sortie: Désordre de la structure métallique avec présence de débris et de corrosion....
  ── [ 31] 3 — . Déformation ou absence de grille ou de caillebot
     133.3s · 0.1 tok/s · sortie: Désordre de corrosion sur la structure métallique....
  ── [ 36] 4 — Les éclats résultant d’un choc
     198.1s · 0.0 tok/s · sortie: Désordre de fissuration sur la structure en béton....
  ── [100] 5 — Affaissement du corps du trottoir
     203.2s · 0.1 tok/s · sortie: Désordre de la chaussée : fissures et dégâts sur le revêtement routier....
  ── [117] 7 — Écoulement en intrados de tablier
     193.6s · 0.0 tok/s · sortie: Désordre de la couverture sous une voie publique....
  ── [143] 8 — Altération du solin
     76.2s · 0.1 tok/s · sortie: Désordre de la ligne droite sur la route....
  ── [179] 8 — Ouverture anormale des joints de trottoir (blocage
     199.2s · 0.1

---
## 5 — Scores BLEU & ROUGE-L + Baseline zero-shot

La **baseline zero-shot** (sortie vide) représente le score plancher — ROUGE-L = 0.  
Tous les modèles doivent dépasser ce plancher pour être pertinents.

In [ ]:
# Load GPU results for scoring (primary device)
try:
    results = results_all.get('cuda', results_all.get('cpu', []))
    assert results
    print(f'✓ Résultats depuis results_all ({len(results)} modèles)')
except Exception:
    rp = results_path.replace('.json', '_cuda.json')
    if not os.path.exists(rp):
        rp = results_path.replace('.json', '_cpu.json')
    with open(rp) as f:
        results = json.load(f)
    print(f'✓ Rechargé depuis {rp}')

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module

smoother = SmoothingFunction().method1
r_scorer  = rs_module.RougeScorer(['rougeL'], use_stemmer=False)

def bleu(ref, hyp):
    if not hyp.strip(): return 0.0
    return round(sentence_bleu(
        [ref.lower().split()], hyp.lower().split(),
        weights=(0.5, 0.5), smoothing_function=smoother,
    ), 4)

def rouge_l(ref, hyp):
    if not hyp.strip(): return 0.0
    return round(r_scorer.score(ref, hyp)['rougeL'].fmeasure, 4)

score_rows, avg_rows = [], []
avg_rows.append({
    'Modèle':        '— Baseline (vide) —',
    'BLEU moyen':    0.0,
    'ROUGE-L moyen': 0.0,
    'VRAM (GB)':     '—',
    'Temps moy.(s)': '—',
})

for r in results:
    if r['status'] != 'success': continue
    bleus, rouges = [], []
    for o in r['outputs']:
        b  = bleu(o['reference'], o['output'])
        rl = rouge_l(o['reference'], o['output'])
        bleus.append(b); rouges.append(rl)
        score_rows.append({
            'Modèle':  r['model_id'],
            'Image':   f'Img {o["idx"]} ({o["fiche"]})',
            'Famille': o['family'],
            'BLEU':    b,
            'ROUGE-L': rl,
        })
    avg_rows.append({
        'Modèle':        r['model_id'],
        'BLEU moyen':    round(sum(bleus)  / len(bleus),  4),
        'ROUGE-L moyen': round(sum(rouges) / len(rouges), 4),
        'VRAM (GB)':     r.get('vram_gb', '—'),
        'Temps moy.(s)': r['avg_time_s'],
    })

df_scores = pd.DataFrame(score_rows)
df_avg    = pd.DataFrame(avg_rows).sort_values('ROUGE-L moyen', ascending=False)

print('── BLEU & ROUGE-L par modèle et par image ───────────────────────────')
display(df_scores.to_string(index=False))
print('\n── Scores moyens sur 10 images (trié par ROUGE-L) ───────────────────')
display(df_avg.to_string(index=False))

df_scores.to_csv(os.path.join(RESULTS_DIR, 'bleu_rouge_scores_v5.csv'), index=False)
df_avg.to_csv(   os.path.join(RESULTS_DIR, 'bleu_rouge_avg_v5.csv'),    index=False)
print(f'\n✓ CSV sauvegardés dans {RESULTS_DIR}')


---
## 6 — Analyse qualitative : structure ITSEOA

Vérifie si les modèles produisent spontanément les 3 composantes du commentaire ITSEOA
(Causes / Gravité / Suites à donner) **sans fine-tuning**.

⚠ Ce n'est pas un critère de scoring — c'est une observation qualitative.  
On s'attend à ce qu'aucun modèle ne produise la structure ITSEOA complète zero-shot,
ce qui justifie la nécessité du fine-tuning en Phase 1.

In [ ]:
print('── Détection de la structure ITSEOA (zero-shot) ─────────────────────')
print('   cause=présence de causes probables | gravité=niveau de sévérité | suites=recommandations')
print()

itseoa_rows = []
for r in results:
    if r['status'] != 'success': continue
    cause_hits   = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('cause',   False))
    gravite_hits = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('gravité', False))
    suites_hits  = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('suites',  False))
    n = len(r['outputs'])
    itseoa_rows.append({
        'Modèle':            r['model_id'],
        'Cause (/10)':       cause_hits,
        'Gravité (/10)':     gravite_hits,
        'Suites (/10)':      suites_hits,
        'Score struct. /30': cause_hits + gravite_hits + suites_hits,
    })

df_itseoa = pd.DataFrame(itseoa_rows).sort_values('Score struct. /30', ascending=False)
display(df_itseoa.to_string(index=False))

print()
print('Note : même avec un score struct. élevé, les termes détectés sont des approximations')
print('génériques — pas la terminologie ITSEOA spécifique. Le fine-tuning Phase 1 est nécessaire.')
df_itseoa.to_csv(os.path.join(RESULTS_DIR, 'itseoa_structure_v5.csv'), index=False)

---
## 7 — Affichage des sorties brutes

Comparaison des sorties de chaque modèle sur une image représentative par famille.

In [ ]:
# Show one representative image per family
shown_families = set()
representative = []
for s in test_samples:
    if s['family'] not in shown_families:
        representative.append(s)
        shown_families.add(s['family'])

for s in representative:
    idx = s['idx']
    print(f'\n{"="*70}')
    print(f'IMAGE {idx} | Fiche {s["fiche"]} | {s["family"]}')
    print(f'{s["desc"]}')
    print(f'{"="*70}')
    thumb = s['image'].copy(); thumb.thumbnail((350, 350)); display(thumb)

    print('\n── COMMENTAIRE EXPERT (référence) ──────────────────────────────────')
    print(textwrap.fill(s['commentary'][:500], width=70))
    if len(s['commentary']) > 500: print('[...]')

    for r in sorted(results, key=lambda x: x.get('leaderboard_rank') or 99):
        if r['status'] != 'success': continue
        oe = next((o for o in r['outputs'] if o['idx'] == idx), None)
        if not oe: continue
        rank_str = f'rank #{r["leaderboard_rank"]}' if r.get('leaderboard_rank') else 'addendum/exclu'
        rl = rouge_l(oe['reference'], oe['output'])
        print(f'\n── {r["model_id"]} [{rank_str}] · ROUGE-L={rl:.3f} ──────────────────────')
        print(textwrap.fill(oe['output'][:500], width=70))
        if len(oe['output']) > 500: print('[...]')

---
## 8 — Scores de Référence Publiés

In [ ]:
bench = pd.DataFrame({
    'Modèle':             ['InternVL3-2B',  'Qwen2.5-VL-3B', 'Qwen3-VL-2B', 'InternVL3.5-2B', 'SmolVLM2-2.2B', 'Gemma3-4B', 'Qwen2.5-Omni-3B', 'LFM2.5-VL'],
    'Params':             ['2B',            '3B',             '2B',           '2.35B',          '2.2B',           '4B',        '3B',               '4B?'],
    'Licence':            ['MIT',           'Apache 2.0',     'Apache 2.0',   'Apache 2.0',     'Apache 2.0',     'Gemma TOS', 'qwen-research',    'LFM 1.0'],
    'Rank (leaderboard)': [10,              11,               '—',            '—',              '—',              '—',         '—',                '—'],
    'Avg OVLMboard':      [64.5,            64.5,             '—',            '—',              '—',              '—',         '—',                '—'],
    'LF template':        ['intern_vl',     'qwen2_5_vl',     'qwen3_vl',     'intern_vl',      '—',              '—',         '—',                '—'],
    'Source':             ['Leaderboard',   'Leaderboard',    'Addendum',     'Addendum',       'Addendum',       'Exclu',     'Exclu',            'Exclu'],
})
print('── Scores publiés (Open VLM Leaderboard + rapports officiels, Avril 2026) ──')
display(bench.to_string(index=False))
bench.to_csv(os.path.join(RESULTS_DIR, 'published_benchmarks_v5.csv'), index=False)

---
## 9 — Décision Finale (seulement sur les modèles retenus)

| Critère | Poids | Justification |
|---------|-------|---------------|
| ROUGE-L moyen (10 images ITSEOA) | 35% | Mesure directe sur notre tâche |
| Hardware (VRAM + vitesse) | 25% | Critique pour fine-tuning T4 |
| Langue française | 20% | Données d'entraînement publiées |
| Fine-tuning LLaMA-Factory | 20% | Template confirmé manuellement |

Note : le score leaderboard public (25% dans v3) a été retiré car il favorisait
Qwen2.5-VL-3B sur des benchmarks génériques non représentatifs de notre tâche ITSEOA.

In [ ]:
# Use GPU results for final scoring
try:
    results = results_all.get('cuda', results_all.get('cpu', []))
except NameError:
    rp = results_path.replace('.json', '_cuda.json')
    if not os.path.exists(rp):
        rp = results_path.replace('.json', '_cpu.json')
    with open(rp) as f:
        results = json.load(f)

# Seuls les modèles avec template LLaMA-Factory (les 4 retenus) sont considérés pour la décision finale
candidate_models = [m for m in MODEL_REGISTRY if m['lf_tpl'] is not None]

def score_rouge_criterion(model_id):
    for r in results:
        if r['model_id'] == model_id and r['status'] == 'success':
            rls = [rouge_l(o['reference'], o['output']) for o in r['outputs']]
            return round((sum(rls) / len(rls)) / 0.30 * 35, 1)
    return 0.0


def score_french(model_id):
    if 'Qwen'     in model_id: return 18.0
    if 'InternVL' in model_id: return 15.0
    return 8.0


def score_ft(model_id):
    return 20.0  # All 4 models have confirmed LF templates


def score_hardware(model_id):
    for r in results:
        if r['model_id'] == model_id and r['status'] == 'success':
            vram = r['vram_gb'] or 15
            spd  = r['avg_tok_per_s'] or 1
            vram_score  = max(0, (5.0 - vram) / 5.0 * 12.5)
            speed_score = min(spd / 8.0, 1.0) * 12.5
            return round(vram_score + speed_score, 1)
    return 0.0


rows_final = []
for entry in candidate_models:
    mid = entry['id']
    rs  = score_rouge_criterion(mid)
    fs  = score_french(mid)
    fts = score_ft(mid)
    hs  = score_hardware(mid)
    rows_final.append({
        'Modèle':          mid,
        'Rank':            entry['rank'],
        'ROUGE-L /35':     rs,
        'Matériel /25':    hs,
        'Français /20':    fs,
        'Fine-tuning /20': fts,
        'TOTAL /100':      round(rs + hs + fs + fts, 1),
    })

df_final     = pd.DataFrame(rows_final).sort_values('TOTAL /100', ascending=False)
winner       = df_final.iloc[0]['Modèle']
winner_total = df_final.iloc[0]['TOTAL /100']
winner_rank  = df_final.iloc[0]['Rank']
winner_tpl   = next(m['lf_tpl'] for m in candidate_models if m['id'] == winner)

print('── Scores pondérés finaux (candidats retenus uniquement) ───────────────────────────')
display(df_final.to_string(index=False))

print(f'\n{"="*60}')
print(f'  MODÈLE RETENU  : {winner}')
print(f'  Leaderboard    : #{winner_rank}')
print(f'  Score pondéré  : {winner_total}/100')
print(f'  LF template    : {winner_tpl}')
print(f'{"="*60}')

decision = {
    'selected_model':       winner,
    'leaderboard_rank':     str(winner_rank),
    'weighted_score':       float(winner_total),
    'lf_template':          winner_tpl,
    'scores_detail':        df_final.to_dict('records'),
    'scoring_weights': {
        'rouge_l':      '35% — mesure directe sur 10 images ITSEOA',
        'hardware':     '25% — VRAM + vitesse, critique pour fine-tuning T4',
        'french':       '20% — données d\'entraînement publiées',
        'finetuning':   '20% — template LLaMA-Factory confirmé manuellement',
        'leaderboard':  '0%  — retiré (benchmarks génériques non représentatifs)',
    },
    'methodology': (
        'Phase initiale : Open VLM Leaderboard top 15 (<=4B), filtré par '
        'disponibilité HF, interface standard, support LLaMA-Factory, licence Apache 2.0 ou MIT. '
        'Addendum : révision systématique LLaMA-Factory vs sorties HuggingFace Mars-Avril 2026. '
        'Évaluation zero-shot sur 10 images ITSEOA stratifiées par famille de désordre. '
        'Extension v5 : évaluation de tous les modèles mentionnés (8 modèles).'
    ),
    'excluded_models': [
        {'id': 'SmolVLM2-2.2B',   'reason': 'Pas de template LLaMA-Factory (évalué mais non retenu)'},
        {'id': 'Gemma3-4B',       'reason': '4B à la limite VRAM T4 ; licence Gemma Terms of Use ; non-VLM'},
        {'id': 'Qwen2.5-Omni-3B', 'reason': 'Licence qwen-research (recherche uniquement)'},
        {'id': 'Qwen3.5-VL',      'reason': "N'existe pas : Qwen3.5 est texte uniquement"},
        {'id': 'LFM2.5-VL',       'reason': 'Licence LFM 1.0 (restreint < 10M$ CA) ; path HF incertain'},
    ],
    'itseoa_note': (
        'Aucun modèle ne produit la structure ITSEOA complète (Causes/Gravité/Suites) '
        'zero-shot. Les occurrences détectées sont des termes génériques, pas la '
        'terminologie ITSEOA spécifique. Ceci confirme la nécessité du fine-tuning Phase 1.'
    ),
    'experimental_results': results,
}

out_path = os.path.join(RESULTS_DIR, 'final_decision_v5.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)
df_final.to_csv(os.path.join(RESULTS_DIR, 'model_weighted_scores_v5.csv'), index=False)

print(f'\n✓ Décision sauvegardée : {out_path}')
print(f'Prochain jalon : Phase 1 — Fine-tuning QLoRA avec LLaMA-Factory (template: {winner_tpl})')

---
## Résumé / Summary

**Fichiers produits dans `{CID_BASE}/phase0/` (version v5) :**
- `model_selection_results_v5.json` — résultats expérimentaux complets (tous les modèles)
- `bleu_rouge_scores_v5.csv` — scores BLEU/ROUGE-L par modèle et par image
- `bleu_rouge_avg_v5.csv` — moyennes par modèle + baseline zero-shot
- `itseoa_structure_v5.csv` — détection structure Causes/Gravité/Suites
- `published_benchmarks_v5.csv` — scores de référence publiés
- `model_weighted_scores_v5.csv` — scores pondérés finaux (uniquement 4 candidats retenus)
- `final_decision_v5.json` — décision documentée complète

*Ces résultats confirment le choix du modèle pour Phase 1 après évaluation de tous les modèles mentionnés.*